# HEADS — Train/Test Split + Multiple Model Variants

This notebook:
1) Loads `data/raw/cybersecurity.csv`
2) Builds anomaly scores (Transformer AE + GraphSAGE)
3) Creates **train/test split** and saves:
   - `data/test/test_v1.csv` (random stratified)
   - `data/test/test_v2.csv` (time-based)
   - `data/test/test_v3.csv` (bootstrap sample)
4) Trains **multiple XGBoost variants** and saves into `models/variants/`:
   - `xgb_v1.json`, `xgb_v2.json`, `xgb_v3.json`
   - plus `tabular_featurizer.joblib`

After running, the dashboard can select different **test instances** and (optionally) different **model variants**.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from xgboost import XGBClassifier
import joblib


In [2]:
DATA_PATH = Path("data/raw/cybersecurity.csv")
MODELS_DIR = Path("models")
VAR_DIR = MODELS_DIR / "variants"
VAR_DIR.mkdir(parents=True, exist_ok=True)

TEST_DIR = Path("data/test")
TEST_DIR.mkdir(parents=True, exist_ok=True)

PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cpu'

In [3]:
REQUIRED_COLS = [
    "timestamp","src_ip","dst_ip","src_port","dst_port","protocol",
    "bytes_sent","bytes_received","user_agent","url","is_internal_traffic"
]

def extract_url_host(url: str) -> str:
    if not url or str(url).lower()=="nan":
        return ""
    m = re.match(r"^https?://([^/]+)/?", str(url))
    return m.group(1).lower() if m else ""

def load_and_prepare(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing=[c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    for c in ["src_port","dst_port","bytes_sent","bytes_received"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    df["protocol"] = df["protocol"].astype(str).fillna("UNK")
    df["user_agent"] = df["user_agent"].astype(str).fillna("UNK")
    df["url"] = df["url"].astype(str).replace("nan","").fillna("")
    df["is_internal_traffic"] = df["is_internal_traffic"].astype(bool).astype(int)

    if "label" in df.columns:
        df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    if "attack_type" in df.columns:
        df["attack_type"] = df["attack_type"].astype(str).fillna("unknown")

    df["hour"] = df["timestamp"].dt.hour.fillna(0).astype(int)
    df["dow"] = df["timestamp"].dt.dayofweek.fillna(0).astype(int)
    df["url_host"] = df["url"].apply(extract_url_host)
    df["bytes_total"] = (df["bytes_sent"] + df["bytes_received"]).astype(int)
    df["is_web"] = df["dst_port"].isin([80,443]).astype(int)

    return df.sort_values(["src_ip","timestamp"], kind="mergesort").reset_index(drop=True)

assert DATA_PATH.exists(), DATA_PATH
df = load_and_prepare(DATA_PATH)
df.head()


,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type,hour,dow,url_host,bytes_total,is_web
0,2025-10-30 03:45:00,1.109.200.95,220.104.40.191,13936,80,TCP,19943,31111,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://portal.example.org/.env?id=131427,1,0,benign,3,3,portal.example.org,51054,1
1,2025-10-11 14:06:00,1.114.12.59,154.97.140.204,21,22,TCP,31877,42581,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,,0,0,benign,14,5,,74458,0
2,2025-12-16 07:12:00,1.120.229.133,251.212.54.33,44201,53,TCP,9675,31559,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://api.service.io/test.php?id=582634,0,0,benign,7,1,api.service.io,41234,0
3,2025-10-26 10:56:00,1.122.157.112,127.149.219.114,80,80,TCP,10003,11137,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,,0,1,brute-force,10,6,,21140,1
4,2025-12-01 03:26:00,1.136.27.125,233.219.48.52,47593,80,TCP,4359,17846,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://app.company.in/config.php?id=602562,0,0,benign,3,0,app.company.in,22205,1


## 1) Train Transformer AE (temporal_score)


In [4]:
SEQ_LEN=20
AE_EPOCHS=3
AE_BATCH=256
AE_LR=3e-4

ae_cols = ["src_port","dst_port","bytes_sent","bytes_received","bytes_total","hour","dow","is_web","is_internal_traffic"]

class SeqDataset(Dataset):
    def __init__(self, X): self.X = X.astype(np.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return torch.from_numpy(self.X[i])

class TransformerAE(nn.Module):
    def __init__(self, feat_dim, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Linear(feat_dim, d_model)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        dec = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerEncoder(dec, num_layers=max(1,layers))
        self.proj_out = nn.Linear(d_model, feat_dim)
    def forward(self, x):
        z = self.proj_in(x)
        z = self.encoder(z)
        z = self.decoder(z)
        return self.proj_out(z)

def build_sequences(df, cols, seq_len):
    counts = df.groupby("src_ip").size()
    eff = max(3, min(seq_len, int(counts.max())))
    X_list=[]; idx=[]
    for _, g in df.groupby("src_ip", sort=False):
        g=g.sort_values("timestamp")
        if len(g) < eff: 
            continue
        X = g[cols].to_numpy(float)
        for i in range(eff-1, len(g)):
            X_list.append(X[i-eff+1:i+1]); idx.append(g.index[i])
    if len(X_list) < 50:
        g = df.sort_values("timestamp")
        X = g[cols].to_numpy(float)
        X_list=[]; idx=[]
        for i in range(eff-1, len(g)):
            X_list.append(X[i-eff+1:i+1]); idx.append(g.index[i])
    return np.stack(X_list), np.array(idx), eff

X_seq, idx_last, eff = build_sequences(df, ae_cols, SEQ_LEN)
ae = TransformerAE(len(ae_cols)).to(DEVICE)

dl = DataLoader(SeqDataset(X_seq), batch_size=AE_BATCH, shuffle=True)
opt = torch.optim.AdamW(ae.parameters(), lr=AE_LR)
loss_fn = nn.MSELoss()

for ep in range(AE_EPOCHS):
    ae.train(); tot=0.0
    for x in dl:
        x=x.to(DEVICE)
        xh=ae(x)
        loss=loss_fn(xh,x)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss.item())*x.size(0)
    print(f"AE epoch {ep+1}/{AE_EPOCHS} loss={tot/len(dl.dataset):.6f}")

with torch.no_grad():
    ae.eval()
    x = torch.tensor(X_seq, dtype=torch.float32, device=DEVICE)
    xh = ae(x)
    scores = torch.mean((xh-x)**2, dim=(1,2)).detach().cpu().numpy()

df["temporal_score"]=0.0
df.loc[idx_last,"temporal_score"]=scores

torch.save(ae.state_dict(), MODELS_DIR/"transformer_ae.pt")
MODELS_DIR/"transformer_ae.pt"


AE epoch 1/3 loss=5751744291696.406250
AE epoch 2/3 loss=5751744154923.682617
AE epoch 3/3 loss=5751744548522.632812


WindowsPath('models/transformer_ae.pt')

## 2) Train GraphSAGE (relational_score)


In [5]:
GNN_EPOCHS=3
GNN_LR=1e-3

def build_graph(df):
    node_map={}; node_types=[]; edges=[]
    def get_node(t,key):
        k=(t,str(key))
        if k in node_map: return node_map[k]
        nid=len(node_map); node_map[k]=nid; node_types.append(t); return nid
    for _,r in df.iterrows():
        a=get_node("actor", r["src_ip"])
        b=get_node("resource", r["dst_ip"])
        edges.append((a,b))
        if r.get("url_host",""):
            edges.append((a, get_node("host", r["url_host"])))
        edges.append((a, get_node("proto", r["protocol"])))
    edge_index=np.array(edges, dtype=np.int64).T
    types_sorted=sorted(set(node_types)); t2i={t:i for i,t in enumerate(types_sorted)}
    type_ids=np.array([t2i[t] for t in node_types], dtype=np.float32)
    deg=np.zeros(len(node_map), dtype=np.float32)
    for u,v in edges: deg[u]+=1; deg[v]+=1
    x=np.stack([type_ids, np.log1p(deg)], axis=1)
    data=Data(x=torch.tensor(x, dtype=torch.float32),
              edge_index=torch.tensor(edge_index, dtype=torch.long),
              num_nodes=len(node_map))
    return data, node_map

class SAGE(nn.Module):
    def __init__(self, in_dim, hidden=64, layers=2):
        super().__init__()
        self.convs=nn.ModuleList([SAGEConv(in_dim, hidden)])
        for _ in range(layers-1):
            self.convs.append(SAGEConv(hidden, hidden))
    def forward(self, x, edge_index):
        for c in self.convs:
            x=torch.relu(c(x, edge_index))
        return x

def neg_sample(n,k):
    return torch.stack([torch.randint(0,n,(k,),device=DEVICE),
                        torch.randint(0,n,(k,),device=DEVICE)], dim=0)

data, node_map = build_graph(df)
data=data.to(DEVICE)
gnn=SAGE(in_dim=data.x.size(1)).to(DEVICE)
opt=torch.optim.Adam(gnn.parameters(), lr=GNN_LR)
E=data.edge_index.size(1)

for ep in range(GNN_EPOCHS):
    gnn.train()
    z=gnn(data.x, data.edge_index)
    s,t = data.edge_index
    pos=(z[s]*z[t]).sum(dim=1)
    ne=neg_sample(data.num_nodes, E)
    ns,nt=ne
    neg=(z[ns]*z[nt]).sum(dim=1)
    loss = -torch.mean(torch.log(torch.sigmoid(pos)+1e-9)) - torch.mean(torch.log(torch.sigmoid(-neg)+1e-9))
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"GNN epoch {ep+1}/{GNN_EPOCHS} loss={float(loss):.6f}")

with torch.no_grad():
    gnn.eval()
    z=gnn(data.x, data.edge_index)
    pairs=[]
    for _,r in df.iterrows():
        pairs.append((node_map[("actor", str(r["src_ip"]))], node_map[("resource", str(r["dst_ip"]))]))
    ei=torch.tensor(np.array(pairs, dtype=np.int64).T, device=DEVICE)
    s,t=ei
    score=(z[s]*z[t]).sum(dim=1)
    rel=(-torch.log(torch.sigmoid(score)+1e-9)).detach().cpu().numpy()

df["relational_score"]=rel
torch.save(gnn.state_dict(), MODELS_DIR/"graphsage.pt")
MODELS_DIR/"graphsage.pt"


GNN epoch 1/3 loss=4.993370
GNN epoch 2/3 loss=4.425826
GNN epoch 3/3 loss=3.815619


WindowsPath('models/graphsage.pt')

## 3) Create multiple TEST sets (variations)


In [6]:
assert "label" in df.columns, "Need label column to create evaluation splits."

# v1: random stratified
train_v1, test_v1 = train_test_split(df, test_size=0.25, random_state=42, stratify=df["label"])
test_v1.to_csv(TEST_DIR/"test_v1_random.csv", index=False)

# v2: time-based (last 20%)
df_time = df.dropna(subset=["timestamp"]).sort_values("timestamp")
cut = int(len(df_time)*0.8)
test_v2 = df_time.iloc[cut:]
test_v2.to_csv(TEST_DIR/"test_v2_time.csv", index=False)

# v3: bootstrap sample
rng=np.random.RandomState(7)
idx=rng.choice(len(df), size=min(5000,len(df)), replace=True)
test_v3 = df.iloc[idx].copy()
test_v3.to_csv(TEST_DIR/"test_v3_bootstrap.csv", index=False)

[(TEST_DIR/"test_v1_random.csv"), (TEST_DIR/"test_v2_time.csv"), (TEST_DIR/"test_v3_bootstrap.csv")]


[WindowsPath('data/test/test_v1_random.csv'),
 WindowsPath('data/test/test_v2_time.csv'),
 WindowsPath('data/test/test_v3_bootstrap.csv')]

## 4) Train multiple XGBoost variants and score test sets


In [7]:
class Featurizer:
    def __init__(self, text_dim=64):
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.hv = HashingVectorizer(n_features=text_dim, alternate_sign=False, norm=None)
    def fit(self, df):
        self.ohe.fit(df[["protocol","url_host","is_internal_traffic"]].astype(str))
        return self
    def transform(self, df):
        num = df[["src_port","dst_port","bytes_sent","bytes_received","bytes_total","hour","dow","is_web"]].to_numpy(float)
        cat = self.ohe.transform(df[["protocol","url_host","is_internal_traffic"]].astype(str))
        txt = (df["url"].fillna("") + " " + df["user_agent"].fillna("")).astype(str).tolist()
        tmat = self.hv.transform(txt).toarray().astype(float)
        return np.hstack([num, cat, tmat])

feat = Featurizer().fit(df)
joblib.dump(feat, MODELS_DIR/"tabular_featurizer.joblib")

X_base = np.hstack([feat.transform(df), df[["temporal_score","relational_score"]].to_numpy(float)])
y_base = df["label"].to_numpy()

variants = [
    {"name":"xgb_v1", "seed":42, "max_depth":5, "lr":0.05, "n":400},
    {"name":"xgb_v2", "seed":7,  "max_depth":4, "lr":0.08, "n":300},
    {"name":"xgb_v3", "seed":99, "max_depth":6, "lr":0.03, "n":500},
]

results = []

for v in variants:
    # split for training this variant
    X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_base, test_size=0.25, random_state=v["seed"], stratify=y_base)
    k = min(5, max(1, int((y_tr==1).sum()-1)))
    sm = SMOTE(random_state=v["seed"], k_neighbors=k)
    X_tr2, y_tr2 = sm.fit_resample(X_tr, y_tr)

    model = XGBClassifier(
        n_estimators=v["n"],
        max_depth=v["max_depth"],
        learning_rate=v["lr"],
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        n_jobs=-1,
        random_state=v["seed"]
    )
    model.fit(X_tr2, y_tr2)

    p = model.predict_proba(X_te)[:,1]
    auc = roc_auc_score(y_te, p) if len(np.unique(y_te))>1 else np.nan
    ap  = average_precision_score(y_te, p) if len(np.unique(y_te))>1 else np.nan
    results.append((v["name"], float(auc), float(ap)))

    out_path = VAR_DIR / f"{v['name']}.json"
    model.save_model(str(out_path))
    print("Saved", out_path)

pd.DataFrame(results, columns=["variant","roc_auc","pr_auc"])


Saved models\variants\xgb_v1.json
Saved models\variants\xgb_v2.json
Saved models\variants\xgb_v3.json


,variant,roc_auc,pr_auc
0,xgb_v1,0.923313,0.639917
1,xgb_v2,0.932896,0.700553
2,xgb_v3,0.915821,0.649482


In [8]:
# Score and save: scored_events.csv + scored test sets for dropdown evaluation
def score_df(df_in: pd.DataFrame, model_path: Path) -> pd.DataFrame:
    model = XGBClassifier()
    model.load_model(str(model_path))
    X = np.hstack([feat.transform(df_in), df_in[["temporal_score","relational_score"]].to_numpy(float)])
    out = df_in.copy()
    out["xgb_proba"] = model.predict_proba(X)[:,1]
    return out

# choose default variant for primary scored_events.csv
default_model = VAR_DIR/"xgb_v1.json"
scored_all = score_df(df, default_model)
scored_all.to_csv(PROC_DIR/"scored_events.csv", index=False)

# score each test set with each variant and save for dropdown
test_files = [
    TEST_DIR/"test_v1_random.csv",
    TEST_DIR/"test_v2_time.csv",
    TEST_DIR/"test_v3_bootstrap.csv"
]

for tf in test_files:
    d = pd.read_csv(tf)
    # ensure derived columns exist (already should)
    for mp in VAR_DIR.glob("xgb_*.json"):
        out = score_df(d, mp)
        out_name = tf.stem + "__" + mp.stem + ".csv"
        out.to_csv(TEST_DIR/out_name, index=False)

print("Saved scored test instances to:", TEST_DIR)


Saved scored test instances to: data\test
